In [0]:
%sql
CREATE CATALOG IF NOT EXISTS water_catalog;

CREATE SCHEMA IF NOT EXISTS water_catalog.bronze;
CREATE SCHEMA IF NOT EXISTS water_catalog.silver;
CREATE SCHEMA IF NOT EXISTS water_catalog.gold;

In [0]:
import requests
import pandas as pd
from time import sleep

print("📥 Récupération API...")

BASE_URL = "https://hubeau.eaufrance.fr/api/v1/qualite_eau_potable/resultats_dis"

all_data = []
page = 1
chunk_size = 1000
MAX_PAGES = 50

while page <= MAX_PAGES:
    params = {
        "date_min_prelevement": "2024-01-01",
        "date_max_prelevement": "2025-12-31",
        "code_departement":69,
        "size": chunk_size,
        "page": page
    }

    response = requests.get(BASE_URL, params=params)
    data = response.json()

    results = data.get("data", [])

    if not results:
        print(f"Fin des données à la page {page}")
        break

    all_data.extend(results)

    print(f"Page {page} récupérée : {len(results)} lignes")

    page += 1
    sleep(0.2)

# ⚠️ IMPORTANT → on NE reset PAS all_data ici
df_final = pd.DataFrame(all_data)

print("Total lignes :", len(df_final))

In [0]:
print("⚡ Conversion en Spark DataFrame...")

df_raw = spark.createDataFrame(df_final)
df_raw.display()

In [0]:
df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("water_catalog.bronze.water_quality")